# 04 — Outline Generation

Turn one saved `VideoTopic` into a timed, structured outline for an
educational short video.

This notebook:

1. Loads a saved topic collection.
2. Selects one topic.
3. Generates a validated outline with the local LLM.
4. Saves the outline as JSON.
5. Previews the hook, sections, visuals, and timing.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.outlines import (
    build_outline_filename,
    find_topic,
    find_topic_file,
    generate_outline,
    save_outline,
)
from educational_shorts.prompts import load_prompt
from educational_shorts.topics import load_topics

print(f"Project root: {PROJECT_ROOT}")

## Configuration

In [ ]:
TOPICS_DIRECTORY = PROJECT_ROOT / "data" / "topics"
OUTLINES_DIRECTORY = PROJECT_ROOT / "data" / "outlines"

# Set this to a specific JSON filename, or leave it as None to use the
# most recently modified topic file.
TOPICS_FILENAME = None

# Select by exact title. Set to None to use TOPIC_INDEX instead.
SELECTED_TOPIC_TITLE = "How Do Bacteria Communicate?"
TOPIC_INDEX = 0

TARGET_SECONDS = 60
SECTION_COUNT = 4
TEMPERATURE = 0.4
GENERATION_SEED = 42

print(f"Topics directory: {TOPICS_DIRECTORY}")
print(f"Outlines directory: {OUTLINES_DIRECTORY}")

## Load topics and select one

In [ ]:
topics_path = find_topic_file(
    topics_directory=TOPICS_DIRECTORY,
    filename=TOPICS_FILENAME,
)

topic_collection = load_topics(topics_path)

selected_topic = find_topic(
    topic_list=topic_collection,
    title=SELECTED_TOPIC_TITLE,
    index=TOPIC_INDEX,
)

print(f"Loaded topics from: {topics_path}")
print(f"Selected topic: {selected_topic.title}")
print(f"Objective: {selected_topic.learning_objective}")

## Load the outline-generation prompt

In [ ]:
outline_system_prompt = load_prompt("outline_generation")

print("Outline-generation prompt loaded.")

## Generate the outline

In [ ]:
video_outline = generate_outline(
    topic=selected_topic,
    system_prompt=outline_system_prompt,
    target_seconds=TARGET_SECONDS,
    section_count=SECTION_COUNT,
    temperature=TEMPERATURE,
    seed=GENERATION_SEED,
)

print(
    f"Generated an outline with {len(video_outline.sections)} body sections."
)
print(
    f"Estimated total duration: "
    f"{video_outline.estimated_total_seconds} seconds"
)

## Save the outline

In [ ]:
output_path = (
    OUTLINES_DIRECTORY
    / build_outline_filename(selected_topic)
)

save_outline(
    outline=video_outline,
    output_path=output_path,
)

print(f"Saved outline to {output_path}")

## Preview

In [ ]:
print(f"TITLE: {video_outline.topic.title}")
print(f"HOOK: {video_outline.hook}")
print()

for index, section in enumerate(video_outline.sections, start=1):
    print(
        f"{index}. {section.section_type.upper()} "
        f"({section.estimated_seconds}s)"
    )
    print(f"   Purpose: {section.purpose}")

    for point in section.key_points:
        print(f"   - {point}")

    print(f"   Visual: {section.visual_direction}")
    print()

print(f"CLOSING TAKEAWAY: {video_outline.closing_takeaway}")
print(
    f"ESTIMATED TOTAL: {video_outline.estimated_total_seconds} seconds"
)